# OpenPlaque — staged left-main and bifurcation tracking

This experiment works only in the **source CCTA volume (series 7)**. It uses the frozen RCA source-volume centerline as a positive calibration reference, then explicitly requires a short left-main trunk with serial coronary-sized lumen QC before attempting preliminary LAD/LCX branches.

Canonical TPV and OpenPlaque PCAT Attenuation are unchanged. LAD/LCX outputs here remain hypotheses until the source-volume QC images are visually accepted.


## Step 1 — Mount Google Drive


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache reuse controls
`True` reuses a valid cache when available; a missing cache is computed and saved. `False` forces that component to be recomputed and its cache refreshed.


In [ ]:
REUSE_SOURCE_EVIDENCE = True
REUSE_RCA_CALIBRATION = True
REUSE_LEFT_OSTIUM = True
REUSE_LEFT_MAIN = True
REUSE_BIFURCATION_BRANCHES = True
REUSE_QC_FIGURES = True
REUSE_REPORT_PACKAGE = True


## Step 3 — Install this branch and lightweight dependencies


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch left-main-bifurcation-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
print('Repository and lightweight dependencies ready.')


## Step 4 — Create the workflow and inspect the cache plan


In [ ]:
from openplaque.left_main_bifurcation_workflow import LeftMainBifurcationWorkflow
reuse = {
    'source_evidence': REUSE_SOURCE_EVIDENCE,
    'rca_calibration': REUSE_RCA_CALIBRATION,
    'left_ostium': REUSE_LEFT_OSTIUM,
    'left_main': REUSE_LEFT_MAIN,
    'bifurcation_branches': REUSE_BIFURCATION_BRANCHES,
    'qc_figures': REUSE_QC_FIGURES,
    'report_package': REUSE_REPORT_PACKAGE,
}
wf = LeftMainBifurcationWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=reuse)
display(wf.cache_status())


## Step 5 — Load source CCTA, reuse/build source evidence, and calibrate against the frozen RCA
The RCA is not being rediscovered. Its already accepted source-volume centerline is used to learn what a genuinely centered coronary lumen looks like in serial orthogonal sections.


In [ ]:
wf.prepare_evidence()
rca_cal = wf.calibrate_rca()
print('RCA calibration summary:')
display(rca_cal)
display(wf.rca_qc_df)


## Step 6 — Generate left-coronary ostium candidates
This step deliberately keeps multiple ostium hypotheses. No ostium is accepted solely from position or HU.


In [ ]:
ostia = wf.detect_left_ostium()
display(ostia.head(12))


## Step 7 — Find and serially validate a short left-main trunk
Each short graph route is rescored using orthogonal lumen geometry. Large contrast chambers and bright boundaries should fail because their connected bright component is too large, off-center, or non-circular.


In [ ]:
left_main = wf.build_left_main()
print('Left-main summary:')
display(wf.left_main_summary)
display(wf.left_main_qc_df)


## Step 8 — From the accepted trunk endpoint, search for a bifurcating LAD/LCX pair
Plaque masks are not used for this geometry. The distal routes must diverge and must independently retain serial coronary-lumen support. They are still labeled **preliminary** pending visual review.


In [ ]:
paths = wf.build_bifurcation_branches()
summary = wf.summary_table()
display(summary)
print('Pair metadata:')
display(wf.branch_pair)


## Step 9 — Create the source-volume QC figures
The decisive review is visual: compare serial left-main cross-sections directly with the frozen RCA reference, then inspect the bifurcation and the distal branch cross-sections.


In [ ]:
figures = wf.plot_qc()
for f in figures:
    print('Saved:', f)


## Step 10 — Package the report-back ZIP


In [ ]:
zip_path = wf.package_report()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LEFT_MAIN_BIFURCATION_REPORT_BACK.zip')


## Step 11 — Report back
After the ZIP is written, return to ChatGPT and say **Retrieve and analyze**. The left-main serial cross-sections are the first acceptance gate; distal LAD/LCX should not be trusted unless that gate passes.
